# native 392px・固定 weak/group 16 検査 pixelcache 診断

CPU・Internet OFF・private 保存実行。Competition と decoder Dataset Version 2 の2 Inputsだけを接続します。教師 CSV と Report は読みません。Gold UID/group と fold0 検証群を除外し、選択 group 全体を保存します。

元 f002 と同じ physical letterbox・3 series × 最大48 adjacent triplets・位置/mask/plane/flags を保持。DINO encoder・学習・予測・submission.csv はありません。約0.989 GiB の raw pixels 上限で、native 一時 memmap＋全体を展開しない圧縮 archive を使います。正解・Public 精度・全件時間はこの診断から判定できません。

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, sys, zipfile
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    raise RuntimeError('Pixel16 diagnostic must never be submitted to the competition')
CODE = Path('/kaggle/working/native392-code')
if CODE.exists():
    raise RuntimeError('Use a fresh saved run; preserve previous outputs')
payload = base64.b64decode('...', validate=True)
if hashlib.sha256(payload).hexdigest() != '06740ee9e8824f11e52e437336bc2336fae53fc4543d8dd9bc7edf9da1690a89':
    raise ValueError('Embedded code ZIP hash mismatch')
expected = {'src/rsna_knee/__init__.py': 'a6f06061df2e890790f44c5ba4af472dd0abc6f465ad48044e6cee53ae0f7751', 'src/rsna_knee/contracts.py': 'aa246fd1a49ecbe88f1795f49904ef9675f119c935e99b9ffd5eee8220cbc71b', 'src/rsna_knee/imaging.py': '07a114888cde19ae0e5799f2de48c6751ec9c4b5637ec1f4f4c220f2911db9ff', 'src/rsna_knee/feature_contracts.py': 'be76ecc28b14491f4d1a0a459794c7f94456e845ce0e46c46ced699bfaf54ed6', 'src/rsna_knee/feature_imaging.py': 'de36a21929b482eee897b12e06bb9964cee59beab6a9bc2b2d44c15ace279546', 'src/rsna_knee/triplet_cache.py': '0b4c666407cb72d7ef3249f479af1e13cca1b4422b18f18cebf3eed2b768f26f', 'bounded_feature_reader.py': '95a906f7e157cf1fa6d722432e6ed913838c6f579ac352418255688a5cf49ee6', 'decoder_runtime.py': '054e6d7e03c602832b37a7ab0a0253aec08847e363afc90e6ba2418e1318210e'}
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if len(archive.infolist()) != len(expected) or set(archive.namelist()) != set(expected):
        raise ValueError('Code member allowlist mismatch')
    validated = {}
    for name in expected:
        target = (CODE / name).resolve()
        if not target.is_relative_to(CODE.resolve()) or '\\' in name or ':' in name:
            raise ValueError('Unsafe code ZIP path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected[name]:
            raise ValueError('Embedded code member hash mismatch')
        validated[name] = data
    CODE.mkdir()
    for name, data in validated.items():
        target = CODE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as stream:
            stream.write(data)
sys.path.insert(0, str(CODE))
sys.path.insert(0, str(CODE / 'src'))
CSV_OPENED = []
def prohibit_teacher_csv(event, args):
    if event == 'open' and args and isinstance(args[0], (str, bytes)):
        path = os.fsdecode(args[0])
        name = Path(path).name.lower()
        if name.endswith('.csv') and '/kaggle/input/' in path.replace('\\', '/'):
            if name != 'train_series.csv':
                raise RuntimeError('Teacher, report, test, or submission CSV read is forbidden in this pixel diagnostic')
            CSV_OPENED.append(path)
sys.addaudithook(prohibit_teacher_csv)
print('Code-only package verified; no encoder or teacher CSV is loaded')


In [ ]:
from decoder_runtime import install_offline_decoders, verify_compressed_phantoms, decoder_inventory, resource_snapshot
from rsna_knee.contracts import dump_json
from rsna_knee.feature_imaging import read_feature_windows
from rsna_knee.triplet_cache import export_pixel_pilot
from bounded_feature_reader import make_bounded_reader
DECODER = Path('/kaggle/input/datasets/rsraki/rsna-dicom-decoders-py313-v1')
if not DECODER.is_dir():
    DECODER = Path('/kaggle/input/rsna-dicom-decoders-py313-v1')
ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
if not ROOT.is_dir():
    ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
if not ROOT.is_dir() or not DECODER.is_dir():
    raise RuntimeError('Attach only Competition and decoder Dataset Version 2')
decoder_install = install_offline_decoders(DECODER / 'wheels', '0db631c509b9878b050d05a328d7479a0671f46db51959ec10b0b2fd73f2f1a5')
decoder_install['synthetic_checks'] = verify_compressed_phantoms(DECODER / 'phantoms', '882a942b99d5fb9a55e7521bcc24ec22edc76fba12f094cffb8afe3100a11dba')
if not all(value['available'] for value in decoder_inventory().values()):
    raise RuntimeError('Required lossless MRI decoders unavailable')
SETTINGS = {'version': 'physical_letterbox_triplet_v1', 'image_size': 392, 'max_series': 3, 'slices_per_series': 48, 'percentiles': [1, 99]}
COHORT = {'schema_version': 'fixed_weak_group_pixel_pilot_v1', 'seed': 20261002, 'held_out_fold': 0, 'n_folds': 5, 'studies': [{'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914', 'group_id': '25a7d9e4e6ae3d1ff213cf0fcadd81ec15601c6c80248ba487ea3bf747364de9', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612', 'group_id': '45c800fdc2b908965f900ce8d02b403fe0f89c8c0ee313bc732228de97ca009a', 'fold': 3, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595', 'group_id': '438af0e9b06ce86493bcabee0d764372fc9a5c8b19f9967aa7041d7b38d55e01', 'fold': 3, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397', 'group_id': '41e20d12cbbea48549f08372b31cc714b0da5c3085dda7a3ac5427b35d21d479', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216', 'group_id': '2439603d8cf8b97e2c4e4a6d1cdc26374faea3da22bc6508bb645fd72da3beb4', 'fold': 4, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737', 'group_id': '3746df8f39b7a425338cceca976e164d640f8a67ecd07bfc162ead85b43a2b1b', 'fold': 3, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707', 'group_id': '0555711065567d144c564a378d60535c375a4b06ea6282eadab843237942eba9', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716', 'group_id': '606d0d4a619fe49afb3356a58c5a6b04cff970ef49836c674fced3a2ccfb3600', 'fold': 3, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431', 'group_id': '3762b22aaec7cf7db5be5e65cd8f5bc2c7058a2ee39ded3ebbbf52cbcf1a34f6', 'fold': 3, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733', 'group_id': '70ef628b1b5ca9a7996d5eb8220d66b92040186bc039ce6b4d1bb5f8b51c2080', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066', 'group_id': '84abd26168be9349ca0f357c7974766500e0a72a64c1c6a585265dd571b2fadf', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984', 'group_id': '6b5b75da9d287fa9f7c79028415e0cd9f5278226feadd6d2bf792fb52f32af7e', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589', 'group_id': '40eafb3f9e263fb212a98ed0c0e51fab28b5dc45c262cf63559e4be59fd317be', 'fold': 2, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286', 'group_id': '6057489f3bb2336728c7cc0b50eff42ce4cd982e1e38bd7c1121f945061e4119', 'fold': 4, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843', 'group_id': 'f4aa248a7838a136146a9262b1e38639acbd0bcf8340c06de916012562538b0b', 'fold': 1, 'source': 'report_weak'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325', 'group_id': '7d8d0444e49ca27948a400f41e736e78d7be11780ac1b69d48946e1f673e62d7', 'fold': 1, 'source': 'report_weak'}], 'gold_ids': ['1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149', '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269', '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052', '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580', '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238', '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935', '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840', '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756', '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604', '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581', '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945', '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819', '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973', '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492', '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803', '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891', '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903', '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309', '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192', '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711', '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378', '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404', '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645', '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465', '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924', '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608', '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550', '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911', '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945', '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742', '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683', '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500', '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125', '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187', '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274', '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132', '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450', '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363', '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710', '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395', '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236', '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910', '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039', '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327', '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774', '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002', '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921', '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220', '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995', '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190', '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003', '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828', '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254', '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037', '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435', '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767', '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312', '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662'], 'gold_groups': ['03c02ad767162c8f6b1c6702edb7ae48e5ccea9536d63a987d2b5b5799a3332a', '07b92def36d27b2fe6a5879fdee2ebe57f953cfe3b3f68b8ebe1d7d6f1d8a0ed', '098a599ad6910a6f042824e2b734c94d3de9a67bfd68b23db17cc619793a615e', '10386a034080af528639531309d08c3661ee1650b76b76af1d20913c86ec045f', '1e4b81f1477e39328c974a21ef9119fbdf1e760ac92498ea7afd8110ffdee8b0', '272de5042e60aaa83e087121e12e597b3a7df8df6a4da76b395f97c7ce80e3ef', '2901d9ea1a478fb24761580410720a90459860893d83ec972503a65a3919c5c0', '31e02b9e8aaed97f5ea2d06acfb529663b6a49a4e8074f8dbe90da353b7e6dcb', '3205200df8c482bcce30610cd9361e254d77f60758744c10ea63b550ca21de75', '3f7449f6a4a3e22cf0b101693b6307c581d8531182a1b94d5d0fd86511a8f9b9', '472cd34a74d5cc26b68de0ff178cc0493739b218e3319146026c87ebb11fdb64', '4c7e2b92fbb2d4ebad2296e22030286ce5b30ae1f8363c6ae1f67a4e126606bf', '4e6a7f6f1773a05e4adfd5ec4564c8c1167375a36107b9c0463bfea14e96be47', '5331dc8b90331eacde40ea83a740d12fb190ee67578de7df910b28b7dd2c2867', '551006b1e03d09db30866bc6f10308dcf3f61892a148b7f946f2e98044a8d177', '554d123984b6110b7c83c3a5cd8433384354ba1f86a47ffa2f1332c2a47679ad', '56b51a810d7c6a0ee1080cb088cdc969491790e45d709c1fb28c81c09e2a6f8d', '59bc3f5b594afd7d1133b2cd3068bcc02c44745bd2b10f9a07b85813d36f5b6b', '5c976010435770520cefa8e364d991e3f516dbb37c2b8b976a4dc0950c56830c', '5ffc80b393a2931f13476e229bbaf7a176dff2c0fe5922b4274238e34fc88631', '66b5496d2175a62167a8aee6e5583691de2fb7d9172b6db548eabbd13564f011', '67d6014d792c8ee8c468eb9730168461ed38a3bdbd6cc5ad065340c5dd65c5f3', '6f823188d3ab272da5fdc7e0d0522504b33ca8650ad7c8abf53ce3dad1045a79', '6fbfb26274ca66942ef9cc1bfb2716152e29e5729a7f5f1151d20a1e1b0300c8', '70b7097ec699fd1fe4ed8f386d5a9b3530c45ba66b34cae9147e95587c1fbe2c', '715eceeb06e2af5006fca1a7dcc706d135aa10f3f4f5bae8201afda904255fd3', '726f60971f4c11da52b1086efd27529802f53dad3a0df37dc7c46bce8068790a', '7e361b918a7ab4239bfa95b877bd8842d7618232372e8b83def6ddd99d59d07b', '7e474a63863eb9050c166c8f3c87cbeeaa5776bf2292c3e33befb291539d6581', '7fbb2a5a8b0ff546b2d542acde3c51fd482a0e95a6627909e12e1b0f1acff59b', '80be32efbd79adf23b7127185880c18c9588afd466e7c739cc46e5f313e1f27a', '8d4c4862e9a8345e366099940ecaf6f77cd699da9dc695d3ba4a5fa331b510a5', '916279b0b2496d0de8e83840b043e42441420ff84b27c8ba65e940777ec18889', '96a14a2a3ca707a0f24fd6943510cfc8e8df96894aa432d3cd7ac6121f39e218', '9753f49e97f282bbd8c752b56b03624fdd075a721aa0e5569f2fd07168498374', '9d3ddff3ed1faa400797d632793fd911bfc1e33e3e5dc5d2fb6eede7f3aaa9d9', 'a5ea720fb283ecd27b7775083565817537adc1be2a5791a0caaa0254fa0a4639', 'a6f448aa5d1794d2a981d582e545e576fc53d76de82b33484f88f13a3a1d6f8a', 'af004b83e27a15fb76938a7afcdc5d1b2c64f722c7e100bdbd714239b223f6d3', 'b635be46ef2ffd4312cdd494006bfc0868c34b37cb1b6a6311275bccc721e621', 'bee8dca223213d9ede966f30666ec6f17f3a12ea091f177ab08e2896d663a129', 'c6de5ee2f987884b1a769896076153fc5238555cd4abac948cfd724991fb1e8f', 'cca0a43fc967c76ad03b15a5460723367a067522a8950ae6946ad6dbd6da67cb', 'ce58c42bfe08ca3b061b19a2c88c6a6a3e1d2b8ed583542f5aeac8def2dcecb4', 'd0813d9eca8145d4234bcb91700ed4a221382a8ff6674a6c67daa2cf004c890f', 'd4fef01b42c9844ec3694301c1c11ce190b8dc4c02efbb7f060cbea6e39e04dd', 'd555b23f5e2a8a48ec033667f207b86e7b8ddd74ad723a35c0241146c5588a61', 'df0e3ddcdd10d681963b36eaba91eef19013ad12334f1bbe450fcee25a08e047', 'e3cd8ba7c4c33857c7a448e2d95590d4c060703648621e3383ba7d7e956aaf9e', 'e53bf44fb3ecee17f623c73f6efb11e39e96ecbe5c281cd3650272cb19337449', 'ef85bac5388f1cbfc869ab867a793d31d2953ba6cd4fbbc4bd84f9b20132914e', 'f0d3a423cbc768e23a19ea3f8a7117fc25cc75f7cb6b702901adea93a39c2ab4', 'f1edab986aa2d68c2a0fd28be5651059e55b1bc972087214f2fb7059c69f4087', 'f31708002735adf90ab33edc514111355cc829dfb22738b036a48029fd60304b', 'f3fa73618ce468691d988a9569e3503282c23f9449a5ecc810a9f7551d49d3fb', 'fb4a16f220ca8505281589cec0636946ac1551a0661e3b69770ca3ec48ea6230', 'fc803c162744b058b024158bc2c12572197692f0403d072ddc20ec366095158d', 'fe9e6c99f87b84f8934a8a86c9a7c6673407bae73b48202a849cca4527dd53b6'], 'complete_selected_groups': {'0555711065567d144c564a378d60535c375a4b06ea6282eadab843237942eba9': ['1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707'], '2439603d8cf8b97e2c4e4a6d1cdc26374faea3da22bc6508bb645fd72da3beb4': ['1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216'], '25a7d9e4e6ae3d1ff213cf0fcadd81ec15601c6c80248ba487ea3bf747364de9': ['1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914'], '3746df8f39b7a425338cceca976e164d640f8a67ecd07bfc162ead85b43a2b1b': ['1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737'], '3762b22aaec7cf7db5be5e65cd8f5bc2c7058a2ee39ded3ebbbf52cbcf1a34f6': ['1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431'], '40eafb3f9e263fb212a98ed0c0e51fab28b5dc45c262cf63559e4be59fd317be': ['1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589'], '41e20d12cbbea48549f08372b31cc714b0da5c3085dda7a3ac5427b35d21d479': ['1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397'], '438af0e9b06ce86493bcabee0d764372fc9a5c8b19f9967aa7041d7b38d55e01': ['1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595'], '45c800fdc2b908965f900ce8d02b403fe0f89c8c0ee313bc732228de97ca009a': ['1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612'], '6057489f3bb2336728c7cc0b50eff42ce4cd982e1e38bd7c1121f945061e4119': ['1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286'], '606d0d4a619fe49afb3356a58c5a6b04cff970ef49836c674fced3a2ccfb3600': ['1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716'], '6b5b75da9d287fa9f7c79028415e0cd9f5278226feadd6d2bf792fb52f32af7e': ['1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984'], '70ef628b1b5ca9a7996d5eb8220d66b92040186bc039ce6b4d1bb5f8b51c2080': ['1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733'], '7d8d0444e49ca27948a400f41e736e78d7be11780ac1b69d48946e1f673e62d7': ['1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325'], '84abd26168be9349ca0f357c7974766500e0a72a64c1c6a585265dd571b2fadf': ['1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066'], 'f4aa248a7838a136146a9262b1e38639acbd0bcf8340c06de916012562538b0b': ['1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843']}, 'manifest_sha256': {'manifest.json': '976abfe212ce23ad8112f16cb61d2c8172bb837ad50aed7ac88886760fc2cabf', 'weak.csv': '700e3da8016d80fd87bde2d1c189ef755ce5e5f89717ebac44d011c967a8371a', 'gold.csv': 'f86a67cf9c5704c37bbe0d7912b308cd7497a2804918c0cc5e8e2008a0b16bad', 'fold-audit-v2.json': 'bca25c48af792853ff1c19eac8e728462696dd869681bd8a3285dc7fcdbc4e24'}, 'selection': 'SHA256(seed:group_id) ascending; whole groups that fit the fixed 16-study budget', 'targets_used_for_selection': False, 'patient_independence_verified': False}
WORKSPACE = Path('/kaggle/working/native392-native-tmp')
WORKSPACE.mkdir()
OUTPUT = Path('/kaggle/working/native392-pixel16')
reader = make_bounded_reader(WORKSPACE)
result = export_pixel_pilot(ROOT, OUTPUT, SETTINGS, COHORT, reader=reader,
                          qc_reader=read_feature_windows, resource_fn=lambda: resource_snapshot('cpu'))
if list(WORKSPACE.iterdir()):
    raise RuntimeError('Native temporary files were not cleaned')
if (Path('/kaggle/working') / 'submission.csv').exists():
    raise RuntimeError('Unexpected submission artifact')
result.update(decoder_install=decoder_install, temporary_files_remaining=0,
              input_csv_reads=sorted(set(CSV_OPENED)), accelerator='CPU',
              storage_reader='disk-backed native intermediate; exact shared preprocessing QC on native volumes <=128MiB')
dump_json(OUTPUT / 'NATIVE392_PIXEL_PILOT_SUMMARY.json', result)
dump_json('/kaggle/working/NATIVE392_PIXEL_PILOT_SUMMARY.json', result)
print({key: result[key] for key in ('status', 'processed', 'raw_pixel_upper_bytes', 'elapsed_seconds', 'preprocessing_byte_QC_count')})
print(result['archive'])


完了条件: complete=true、processed=16、shared preprocessing byte QC >0、temporary_files_remaining=0、入力 CSV は train_series.csv のみ。native392-pixel16.zip と summary を次段の限定 profiler に使用します。Kaggle 提出には使いません。